In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from google import genai
from google.genai import types

gemini_client = genai.Client(
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            initial_delay=20.0, 
            attempts=3          
        )
    )
) # picks up the API key from the env variable GEMINI_API_KEY

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
assistant = RAGBase(
    index = index,
    llm_client=gemini_client
)

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
from google.genai import types

# Define the schema explicitly
search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
    }
)

search_tool = types.Tool(
    function_declarations=[search_declaration]
)

*
*
*
Conversation History

1. Making a request (query) to the LLM <-- first request
2. LLM decides to invoke Search('with parameters')
3. Getting results as Search() output
4. Sending the results back to the LLM <-- a second request
5. LLM processes the results
6. LLM gives an answer

In [7]:
agent_instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [8]:
from google.genai import types

def make_call(call):
    # Gemini already parsed the args into a dict!
    args = call.args

    if call.name == "search":
        result = search(**args)

    # Gemini expects a specific Content/Part object back
    return types.Content(
        role="user",
        parts=[
            types.Part.from_function_response(
                name=call.name,
                response={"result": result}
            )
        ]
    )

In [14]:
from google.genai import types

def agent_loop(instructions, question, model="gemini-3.6-flash") -> str:

    conversation_history = [
        {'role': 'user', 'parts': [{'text': question}]},
    ]

    conv_iteration = 1
    while True:
        print(f"iteration #{conv_iteration}...")
        has_function_calls = False

        # Call the model with the current conversation history
        response = gemini_client.models.generate_content(
            model = model,
            contents=conversation_history,
            config=types.GenerateContentConfig(
                system_instruction = instructions,
                tools=[search_tool]
            )
        )

        # Append the model's response to the history
        conversation_history.append(response.candidates[0].content)

        for part in response.candidates[0].content.parts:
            
            # Checking if the item is a function call
            if part.function_call:
                print("function_call:", part.function_call.name, part.function_call.args)
                call_output = make_call(part.function_call)
                conversation_history.append(call_output)
                has_function_calls = True

            # Checking if the item is a standard message
            elif part.text:
                print("ASSISTANT:")
                last_answer = part.text
                print(last_answer)

        conv_iteration += 1

        # Exit condition
        if has_function_calls == False:
            break

    return last_answer

In [10]:
agent_loop(agent_instructions, "what's queen gambit?")

iteration #1...
function_call: search {'query': 'queen gambit'}
iteration #2...
function_call: search {'query': 'gambit'}
iteration #3...
function_call: search {'query': 'queen'}
iteration #4...
function_call: search {'query': 'course'}
iteration #5...
ASSISTANT:
This question appears to be off-topic. The course FAQ only contains information regarding the course and its logistics, and does not cover chess or "Queen's Gambit." 

Are there any other areas related to the course that you would like to explore?


'This question appears to be off-topic. The course FAQ only contains information regarding the course and its logistics, and does not cover chess or "Queen\'s Gambit." \n\nAre there any other areas related to the course that you would like to explore?'

In [15]:
agent_loop(agent_instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {'query': 'can I still join the course late registration discovered'}
iteration #2...
function_call: search {'query': 'join course late start late'}
iteration #3...
ASSISTANT:
Yes, you can still join the course! 

If you want to receive a certificate, you will need to submit your project while submissions are still being accepted. You don't need a formal registration confirmation—you can simply start learning and submitting your work while the submission forms are open.

Are there other areas or questions about the course that you would like to explore?


"Yes, you can still join the course! \n\nIf you want to receive a certificate, you will need to submit your project while submissions are still being accepted. You don't need a formal registration confirmation—you can simply start learning and submitting your work while the submission forms are open.\n\nAre there other areas or questions about the course that you would like to explore?"

In [11]:
""" def calculate_gemini_flash_price(response):

    usage = response.usage_metadata
    input_tokens = usage.prompt_token_count
    output_tokens = usage.candidates_token_count

    print("Input Tokens:", input_tokens)
    print("Output Tokens:", output_tokens)

    # Standard pricing rates for gemini-3.6-flash (per million tokens)
    INPUT_PRICE_PER_MILLION = 0.75
    OUTPUT_PRICE_PER_MILLION = 4.50

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost


    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost:": total_cost,
    }


cost_info = calculate_gemini_flash_price(response)
print("Total cost: $", round(cost_info["total_cost"], 8)) """

' def calculate_gemini_flash_price(response):\n\n    usage = response.usage_metadata\n    input_tokens = usage.prompt_token_count\n    output_tokens = usage.candidates_token_count\n\n    print("Input Tokens:", input_tokens)\n    print("Output Tokens:", output_tokens)\n\n    # Standard pricing rates for gemini-3.6-flash (per million tokens)\n    INPUT_PRICE_PER_MILLION = 0.75\n    OUTPUT_PRICE_PER_MILLION = 4.50\n\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\n    total_cost = input_cost + output_cost\n\n\n    return {\n        "input_cost": input_cost,\n        "output_cost": output_cost,\n        "total_cost:": total_cost,\n    }\n\n\ncost_info = calculate_gemini_flash_price(response)\nprint("Total cost: $", round(cost_info["total_cost"], 8)) '